In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import requests
import time
import io
import re
from pathlib import Path

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}



In [68]:
df_italy = pd.read_csv('./data/processed/serie_a_final_standings.csv')

In [69]:
df_italy.head()

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts,Qualification or relegation,season
0,1,Torino,38,28,7,3,104,35,+69,63,NaN,1946/47
1,2,Juventus,38,22,9,7,83,38,+45,53,NaN,1946/47
2,3,Modena,38,21,9,8,45,24,+21,51,NaN,1946/47
3,4,Milan,38,19,12,7,75,52,+23,50,NaN,1946/47
4,5,Bologna,38,15,9,14,42,41,+1,39,NaN,1946/47


### Understanding England - Premier leauge - structure

In [14]:
import pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

def build_url_england(season_start: int) -> str:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    if season_start < 1992:
        page = f"{season_start}%E2%80%93{suffix}_Football_League_First_Division"
    else:
        page = f"{season_start}%E2%80%93{suffix}_Premier_League"
    return f"https://en.wikipedia.org/wiki/{page}"

def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join(str(c) for c in col if c and not str(c).startswith("Unnamed")).strip()
            for col in df.columns
        ]
    else:
        df.columns = df.columns.astype(str)
    return df

def find_standings_table(tables: list[pd.DataFrame], season_label: str) -> pd.DataFrame | None:
    pts_aliases = {"Pts", "Points", "PTS", "Pt", "P"}
    pos_aliases = {"Pos", "Rank", "#", "Position"}

    for t in tables:
        t = flatten_columns(t)
        cols = set(t.columns)
        has_pts = bool(cols & pts_aliases)
        has_pos = bool(cols & pos_aliases)
        if has_pts and has_pos and len(t.columns) >= 6:
            t = t.copy()

            alt_team_col = next(
                (c for c in t.columns if c.startswith("Team") and c != "Team"), None
            )
            if "Team" not in t.columns and alt_team_col is not None:
                t = t.rename(columns={alt_team_col: "Team"})
            elif alt_team_col is not None and "Team" in t.columns and t["Team"].isna().all():
                t["Team"] = t[alt_team_col]

            junk = [
                c for c in t.columns
                if c.startswith("Unnamed") or (c.startswith("Team") and c != "Team")
            ]
            t = t.drop(columns=junk)

            t["season"] = season_label
            return t
    return None

def get_england_table(season_start: int) -> pd.DataFrame | None:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    season_label = f"{season_start}/{suffix}"
    url = build_url_england(season_start)

    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    try:
        tables = pd.read_html(io.StringIO(response.text))
    except Exception as e:
        raise RuntimeError(f"Failed to parse tables from {url}: {e}") from e

    return find_standings_table(tables, season_label)

def scrape_england_seasons(
    start: int = 1963,
    end: int = 2025,
    output_dir: str = "data/processed",
    sleep_seconds: float = 1.5,
) -> pd.DataFrame:
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    all_tables = []
    failed = []

    for year in range(start, end + 1):
        try:
            print(f"{year}: scraping...", end=" ", flush=True)
            t = get_england_table(year)
            if t is not None:
                all_tables.append(t)
                print(f"OK — {len(t)} rows, columns: {list(t.columns)}")
            else:
                print("no standings table found")
                failed.append(year)
        except Exception as e:
            print(f"ERROR — {e}")
            failed.append(year)

        time.sleep(sleep_seconds)

    if not all_tables:
        print("No data collected.")
        return pd.DataFrame()

    df = pd.concat(all_tables, ignore_index=True)

    csv_path = out_path / "england_top_league_standings.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved {len(df)} rows to {csv_path}")

    if failed:
        print(f"Failed seasons: {failed}")

    return df


In [10]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

def build_url_england(season_start: int) -> str:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    if season_start < 1992:
        page = f"{season_start}%E2%80%93{suffix}_Football_League_First_Division"
    else:
        page = f"{season_start}%E2%80%93{suffix}_Premier_League"
    return f"https://en.wikipedia.org/wiki/{page}"

# Step 1: inspect one season from each era
test_seasons = {
    "First Division (old)": 1975,
    "First Division (boundary)": 1991,
    "Premier League (early)": 1993,
    "Premier League (modern)": 2018,
    "Premier League (latest)": 2024,
}

for label, year in test_seasons.items():
    url = build_url_england(year)
    print(f"\n{'='*60}")
    print(f"{label} — {year}: {url}")
    response = requests.get(url, headers=HEADERS, timeout=15)
    tables = pd.read_html(io.StringIO(response.text))
    print(f"  {len(tables)} tables found")
    for i, t in enumerate(tables):
        print(f"  Table {i}: shape={t.shape}, columns={list(t.columns[:6])}...")
    time.sleep(1.5)



First Division (old) — 1975: https://en.wikipedia.org/wiki/1975%E2%80%9376_Football_League_First_Division
  12 tables found
  Table 0: shape=(1, 2), columns=[0, 1]...
  Table 1: shape=(12, 2), columns=[0, 1]...
  Table 2: shape=(22, 11), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 3: shape=(22, 23), columns=['Home \\ Away', 'ARS', 'AST', 'BIR', 'BUR', 'COV']...
  Table 4: shape=(5, 7), columns=['Team', 'Outgoing manager', 'Manner of departure', 'Date of vacancy', 'Position in table', 'Incoming manager']...
  Table 5: shape=(9, 4), columns=['Rank', 'Player', 'Club', 'Goals']...
  Table 6: shape=(2, 2), columns=['vteFootball League First Division', 'vteFootball League First Division.1']...
  Table 7: shape=(17, 2), columns=['vte1975–76 in English football', 'vte1975–76 in English football.1']...
  Table 8: shape=(1, 2), columns=[0, 1]...
  Table 9: shape=(3, 2), columns=[0, 1]...
  Table 10: shape=(2, 2), columns=[0, 1]...
  Table 11: shape=(5, 2), columns=[0, 1]...

First 

In [14]:
# Step 2: verify extraction works for one season per era

def get_england_table(season_start: int) -> pd.DataFrame | None:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    season_label = f"{season_start}/{suffix}"
    url = build_url_england(season_start)

    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    try:
        tables = pd.read_html(io.StringIO(response.text))
    except Exception as e:
        raise RuntimeError(f"Failed to parse tables from {url}: {e}") from e

    return find_standings_table(tables, season_label)


for label, year in test_seasons.items():
    print(f"\n{label} — {year}")
    try:
        t = get_england_table(year)
        if t is not None:
            print(t[["Pos", "Team", "Pld", "Pts"]].head(5).to_string(index=False))
        else:
            print("  no table found")
    except Exception as e:
        print(f"  ERROR: {e}")
    time.sleep(1.5)



First Division (old) — 1975
 Pos                Team  Pld  Pts
   1       Liverpool (C)   42   60
   2 Queens Park Rangers   42   59
   3   Manchester United   42   56
   4        Derby County   42   53
   5        Leeds United   42   51

First Division (boundary) — 1991
 Pos                Team  Pld  Pts
   1    Leeds United (C)   42   82
   2   Manchester United   42   78
   3 Sheffield Wednesday   42   75
   4             Arsenal   42   72
   5     Manchester City   42   70

Premier League (early) — 1993
 Pos                  Team  Pld  Pts
   1 Manchester United (C)   42   92
   2      Blackburn Rovers   42   84
   3      Newcastle United   42   77
   4               Arsenal   42   71
   5          Leeds United   42   70

Premier League (modern) — 2018
 Pos                Team  Pld  Pts
   1 Manchester City (C)   38   98
   2           Liverpool   38   97
   3             Chelsea   38   72
   4   Tottenham Hotspur   38   71
   5             Arsenal   38   70

Premier League (lates

In [21]:
df_england = scrape_england_seasons(start=1946, end=2025)
df_england.head()

1946: scraping... OK — 22 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Relegation', 'season']
1947: scraping... OK — 22 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Relegation', 'season']
1948: scraping... OK — 22 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Relegation', 'season']
1949: scraping... OK — 22 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Relegation', 'season']
1950: scraping... OK — 22 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Relegation', 'season']
1951: scraping... OK — 22 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Relegation', 'season']
1952: scraping... OK — 22 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Relegation', 'season']
1953: scraping... OK — 22 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv',

,Pos,Team,Pld,W,D,L,GF,GA,GAv,Pts,Relegation,season,Qualification or relegation,GD
0,1,Liverpool (C),42,25,7,10,84,52,1.615,57,NaN,1946/47,NaN,NaN
1,2,Manchester United,42,22,12,8,95,54,1.759,56,NaN,1946/47,NaN,NaN
2,3,Wolverhampton Wanderers,42,25,6,11,98,56,1.750,56,NaN,1946/47,NaN,NaN
3,4,Stoke City,42,24,7,11,90,53,1.698,55,NaN,1946/47,NaN,NaN
4,5,Blackpool,42,22,6,14,71,70,1.014,50,NaN,1946/47,NaN,NaN


In [22]:
df_england.isna().sum()

Pos                               0
Team                              0
Pld                               0
W                                 0
D                                 0
L                                 0
GF                                0
GA                                0
GAv                            1031
Pts                               0
Relegation                     1675
season                            0
Qualification or relegation    1028
GD                              660
dtype: int64

In [23]:
df_england[df_england['GD'].isna()]['season'].unique()

<StringArray>
['1946/47', '1947/48', '1948/49', '1949/50', '1950/51', '1951/52', '1952/53',
 '1953/54', '1954/55', '1955/56', '1956/57', '1957/58', '1958/59', '1959/60',
 '1960/61', '1961/62', '1962/63', '1963/64', '1964/65', '1965/66', '1966/67',
 '1967/68', '1968/69', '1969/70', '1970/71', '1971/72', '1972/73', '1973/74',
 '1974/75', '1975/76']
Length: 30, dtype: str

In [24]:
df_england[df_england['GAv'].isna()]['season'].unique()

<StringArray>
[  '1976/77',   '1977/78',   '1978/79',   '1979/80',   '1980/81',   '1981/82',
   '1982/83',   '1983/84',   '1984/85',   '1985/86',   '1986/87',   '1987/88',
   '1988/89',   '1989/90',   '1990/91',   '1991/92',   '1992/93',   '1993/94',
   '1994/95',   '1995/96',   '1996/97',   '1997/98',   '1998/99', '1999/2000',
   '2000/01',   '2001/02',   '2002/03',   '2003/04',   '2004/05',   '2005/06',
   '2006/07',   '2007/08',   '2008/09',   '2009/10',   '2010/11',   '2011/12',
   '2012/13',   '2013/14',   '2014/15',   '2015/16',   '2016/17',   '2017/18',
   '2018/19',   '2019/20',   '2020/21',   '2021/22',   '2022/23',   '2023/24',
   '2024/25',   '2025/26']
Length: 50, dtype: str

In [28]:
df_england.shape

(1691, 14)

In [33]:
df_england.dtypes

Pos                              int64
Team                               str
Pld                              int64
W                                int64
D                                int64
L                                int64
GF                               int64
GA                               int64
GAv                            float64
Pts                             object
Relegation                         str
season                             str
Qualification or relegation        str
GD                                 str
dtype: object

## Fixing Nans

In [34]:
# Recompute GD and GAv from GF and GA (always available)
df_england["GD"] = df_england["GF"] - df_england["GA"]
df_england["GAv"] = (df_england["GF"] / df_england["GA"]).round(3)


In [35]:
df_england.isna().sum()

Pos                               0
Team                              0
Pld                               0
W                                 0
D                                 0
L                                 0
GF                                0
GA                                0
GAv                               0
Pts                               0
Relegation                     1675
season                            0
Qualification or relegation    1028
GD                                0
dtype: int64

In [36]:
df_england[df_england['GD'] == df_england['GD'].max()]

,Pos,Team,Pld,W,D,L,GF,GA,GAv,Pts,Relegation,season,Qualification or relegation,GD
1511,1,Manchester City (C),38,32,4,2,106,27,3.926,100,NaN,2017/18,Qualification for the Champions League group s...,79


In [42]:
df_england[df_england['GAv'] == df_england['GAv'].min()]

,Pos,Team,Pld,W,D,L,GF,GA,GAv,Pts,Relegation,season,Qualification or relegation,GD
1330,20,Derby County (R),38,1,8,29,20,89,0.225,11,NaN,2007/08,Relegation to Football League Championship,-69


In [38]:
85/42

2.0238095238095237

In [43]:
np.sort(df_england['Team'].unique())

array(['Arsenal', 'Arsenal (C)', 'Arsenal[a]', 'Aston Villa',
       'Aston Villa (C)', 'Aston Villa (R)', 'Barnsley (R)',
       'Birmingham City', 'Birmingham City (R)', 'Blackburn Rovers',
       'Blackburn Rovers (C)', 'Blackburn Rovers (R)', 'Blackpool',
       'Blackpool (R)', 'Bolton Wanderers', 'Bolton Wanderers (R)',
       'Bournemouth', 'Bournemouth (R)', 'Bradford City',
       'Bradford City (R)', 'Brentford', 'Brentford (R)',
       'Brighton & Hove Albion', 'Brighton & Hove Albion (R)',
       'Bristol City', 'Bristol City (R)', 'Burnley', 'Burnley (C)',
       'Burnley (R)', 'Cardiff City', 'Cardiff City (R)',
       'Carlisle United (R)', 'Charlton Athletic',
       'Charlton Athletic (O)', 'Charlton Athletic (R)', 'Chelsea',
       'Chelsea (C)', 'Chelsea (R)', 'Coventry City', 'Coventry City (R)',
       'Coventry City[b]', 'Crystal Palace', 'Crystal Palace (R)',
       'Crystal Palace[d] (R)', 'Derby County', 'Derby County (C)',
       'Derby County (R)', 'Everton',

In [44]:
import re

ENGLAND_CITY_MAP = {
    "Arsenal": "London",
    "Aston Villa": "Birmingham",
    "Barnsley": "Barnsley",
    "Birmingham City": "Birmingham",
    "Blackburn Rovers": "Blackburn",
    "Blackpool": "Blackpool",
    "Bolton Wanderers": "Bolton",
    "Bournemouth": "Bournemouth",
    "Bradford City": "Bradford",
    "Brentford": "London",
    "Brighton & Hove Albion": "Brighton",
    "Bristol City": "Bristol",
    "Burnley": "Burnley",
    "Cardiff City": "Cardiff",
    "Carlisle United": "Carlisle",
    "Charlton Athletic": "London",
    "Chelsea": "London",
    "Coventry City": "Coventry",
    "Crystal Palace": "London",
    "Derby County": "Derby",
    "Everton": "Liverpool",
    "Fulham": "London",
    "Grimsby Town": "Grimsby",
    "Huddersfield Town": "Huddersfield",
    "Hull City": "Hull",
    "Ipswich Town": "Ipswich",
    "Leeds United": "Leeds",
    "Leicester City": "Leicester",
    "Leyton Orient": "London",
    "Liverpool": "Liverpool",
    "Luton Town": "Luton",
    "Manchester City": "Manchester",
    "Manchester United": "Manchester",
    "Middlesbrough": "Middlesbrough",
    "Millwall": "London",
    "Newcastle United": "Newcastle",
    "Northampton Town": "Northampton",
    "Norwich City": "Norwich",
    "Nottingham Forest": "Nottingham",
    "Notts County": "Nottingham",
    "Oldham Athletic": "Oldham",
    "Oxford United": "Oxford",
    "Portsmouth": "Portsmouth",
    "Preston North End": "Preston",
    "Queens Park Rangers": "London",
    "Reading": "Reading",
    "Sheffield United": "Sheffield",
    "Sheffield Wednesday": "Sheffield",
    "Southampton": "Southampton",
    "Stoke City": "Stoke-on-Trent",
    "Sunderland": "Sunderland",
    "Swansea City": "Swansea",
    "Swindon Town": "Swindon",
    "Tottenham Hotspur": "London",
    "Watford": "Watford",
    "West Bromwich Albion": "West Bromwich",
    "West Ham United": "London",
    "Wigan Athletic": "Wigan",
    "Wimbledon": "London",
    "Wolverhampton Wanderers": "Wolverhampton",
}

def clean_england_team(name: str) -> str:
    if pd.isna(name):
        return name
    name = re.sub(r'\[.*?\]', '', name)
    name = re.sub(r'\(.*?\)', '', name)
    return name.strip()

df_england["Team"] = df_england["Team"].apply(clean_england_team)
df_england["City"] = df_england["Team"].map(ENGLAND_CITY_MAP)

# Flag any teams not in the map
unmapped = df_england[df_england["City"].isna()]["Team"].unique()
if len(unmapped):
    print(f"Unmapped teams: {unmapped}")


In [45]:
df_england.isna().sum()

Pos                               0
Team                              0
Pld                               0
W                                 0
D                                 0
L                                 0
GF                                0
GA                                0
GAv                               0
Pts                               0
Relegation                     1675
season                            0
Qualification or relegation    1028
GD                                0
City                              0
dtype: int64

In [46]:
df_england['Team'].nunique()

60

In [47]:
np.sort(df_england['Team'].unique())

array(['Arsenal', 'Aston Villa', 'Barnsley', 'Birmingham City',
       'Blackburn Rovers', 'Blackpool', 'Bolton Wanderers', 'Bournemouth',
       'Bradford City', 'Brentford', 'Brighton & Hove Albion',
       'Bristol City', 'Burnley', 'Cardiff City', 'Carlisle United',
       'Charlton Athletic', 'Chelsea', 'Coventry City', 'Crystal Palace',
       'Derby County', 'Everton', 'Fulham', 'Grimsby Town',
       'Huddersfield Town', 'Hull City', 'Ipswich Town', 'Leeds United',
       'Leicester City', 'Leyton Orient', 'Liverpool', 'Luton Town',
       'Manchester City', 'Manchester United', 'Middlesbrough',
       'Millwall', 'Newcastle United', 'Northampton Town', 'Norwich City',
       'Nottingham Forest', 'Notts County', 'Oldham Athletic',
       'Oxford United', 'Portsmouth', 'Preston North End',
       'Queens Park Rangers', 'Reading', 'Sheffield United',
       'Sheffield Wednesday', 'Southampton', 'Stoke City', 'Sunderland',
       'Swansea City', 'Swindon Town', 'Tottenham Hotspur'

In [50]:
e1 = df_england[df_england['Pos'] == 1]
e1.shape

(80, 15)

In [53]:
e1['Team'].value_counts()

Team
Manchester United          18
Liverpool                  16
Arsenal                     9
Manchester City             9
Chelsea                     6
Everton                     4
Wolverhampton Wanderers     3
Leeds United                3
Portsmouth                  2
Tottenham Hotspur           2
Derby County                2
Burnley                     1
Ipswich Town                1
Nottingham Forest           1
Aston Villa                 1
Blackburn Rovers            1
Leicester City              1
Name: count, dtype: int64

In [54]:
i1 = ita_data[ita_data['Pos'] == 1]

In [56]:
i1['Team'].value_counts()

Team
Juventus         30
Milan            16
Inter Milan      16
Torino            4
Napoli            4
Fiorentina        2
Lazio             2
Roma              2
Bologna           1
Cagliari          1
Hellas Verona     1
Sampdoria         1
Name: count, dtype: int64

## Grouping for Teams and Cities

In Italy, we say that three teams (Juventus, Inter and Milan) belonging 2 cities (Torino and Milano) dominates the League tables from 1946 onward. Is the same happening in different leagues/countries? Let's first see what happens in England with Premier League (previously First Division).

## Methodology: Top 4 Counts by Team and City

The core question is whether top-flight football concentrates around a few clubs/cities or is more distributed, and whether that pattern is consistent across leagues (Italy, England, Germany, Spain).

### Function: `compute_top4_counts`

Computes, for each team and each city, how many times they finished in the top 4 positions of the league in a given season. Two definitions of "top 4" are used in parallel to assess the sensitivity of the results to the definition chosen.

### Definitions of "Top 4"

Two definitions are computed and compared:

- **By position (`top4_by_pos`)**: a team is in the top 4 if its official final position (`Pos`) is ≤ 4.
- **By points rank (`top4_by_pts`)**: a team is in the top 4 if its points total is ≥ the 4th highest
  points total in that season. In case of ties on points, all tied teams are included,
  which may result in more than 4 teams qualifying in a given season.

The two definitions can disagree when teams are tied on points but separated by other tiebreakers (goal difference, goals scored, head-to-head). The column `definition_differs` flags seasons where the two definitions yield a different count.


### Output

The function returns two dataframes — one aggregated by team, one by city — with the same structure:

| Column | Description |
|---|---|
| `top4_by_pos` | Number of seasons the team/city finished in the top 4 by official position |
| `top4_by_pts` | Number of seasons the team/city finished in the top 4 by points rank |
| `seasons_in_league` | Total number of seasons the team/city appeared in the top flight |
| `top4_by_pos_pct` | `top4_by_pos` / `seasons_in_league` — normalised rate |
| `top4_by_pts_pct` | `top4_by_pts` / `seasons_in_league` — normalised rate |

Results are sorted alphabetically by default. Percentage columns normalise for teams that were promoted and relegated frequently: a team finishing top 4 in 5 out of 5 seasons is treated differently from one doing so in 5 out of 50.

### Caveats

- **City aggregations with multiple teams** (e.g. London, Manchester, Milan, Madrid) accumulate counts across all their clubs. Raw counts will therefore naturally be higher for these cities. A per-team average within the city will be introduced in a later stage to make the comparison fair.
- **Teams absent from the league** in a given season are excluded from the count for that season. They are not penalised with a zero, but their `seasons_in_league` denominator reflects only the seasons they actually participated.
- **Historical name changes** (e.g. Chievo / ChievoVerona) have been conflated into a single team name prior to this analysis. See the data cleaning section for the full mapping.

In [60]:
import re

def compute_top4_counts(
    df: pd.DataFrame,
    city_col: str = "City",
    team_col: str = "Team",
    pos_col: str = "Pos",
    pts_col: str = "Pts",
) -> tuple[pd.DataFrame, pd.DataFrame]:

    def parse_pts(val):
        if pd.isna(val):
            return pd.NA
        return int(re.sub(r'\[.*?\]', '', str(val)).strip())

    df = df.copy()
    df["_pts_clean"] = df[pts_col].apply(parse_pts)
    df["_pos_clean"] = df[pos_col].apply(
        lambda v: int(re.sub(r'\[.*?\]', '', str(v)).strip()) if pd.notna(v) else pd.NA
    )

    team_records = []
    city_records = []

    for season, group in df.groupby("season"):
        # Top 4 by Pos
        top4_pos = set(group[group["_pos_clean"] <= 4][team_col])

        # Top 4 by Pts
        fourth_pts = group["_pts_clean"].nlargest(4).min()
        top4_pts = set(group[group["_pts_clean"] >= fourth_pts][team_col])

        for _, row in group.iterrows():
            team = row[team_col]
            city = row.get(city_col, None)
            team_records.append({
                "team": team,
                "city": city,
                "season": season,
                "in_top4_by_pos": team in top4_pos,
                "in_top4_by_pts": team in top4_pts,
            })

    season_df = pd.DataFrame(team_records)

    # --- Team aggregation ---
    team_agg = (
        season_df.groupby("team")
        .agg(
            top4_by_pos=("in_top4_by_pos", "sum"),
            top4_by_pts=("in_top4_by_pts", "sum"),
            seasons_in_league=("season", "count"),
        )
        .astype({"top4_by_pos": int, "top4_by_pts": int})
        .sort_index()
    )
    team_agg["top4_by_pos_pct"] = (team_agg["top4_by_pos"] / team_agg["seasons_in_league"]).round(3)
    team_agg["top4_by_pts_pct"] = (team_agg["top4_by_pts"] / team_agg["seasons_in_league"]).round(3)

    # --- City aggregation ---
    city_agg = (
        season_df.groupby("city")
        .agg(
            top4_by_pos=("in_top4_by_pos", "sum"),
            top4_by_pts=("in_top4_by_pts", "sum"),
            seasons_in_league=("season", "count"),
        )
        .astype({"top4_by_pos": int, "top4_by_pts": int})
        .sort_index()
    )
    city_agg["top4_by_pos_pct"] = (city_agg["top4_by_pos"] / city_agg["seasons_in_league"]).round(3)
    city_agg["top4_by_pts_pct"] = (city_agg["top4_by_pts"] / city_agg["seasons_in_league"]).round(3)

    return team_agg, city_agg


In [ ]:
team_counts, city_counts = compute_top4_counts(df_england)
print("=== By Team ===")
display(team_counts)
print("=== By City ===")
display(city_counts)


=== By Team ===


,top4_by_pos,top4_by_pts,seasons_in_league,top4_by_pos_pct,top4_by_pts_pct
team,,,,,
Arsenal,39,43,80,0.488,0.538
Aston Villa,7,7,67,0.104,0.104
Barnsley,0,0,1,0.000,0.000
Birmingham City,0,0,31,0.000,0.000
Blackburn Rovers,3,3,28,0.107,0.107
Blackpool,3,4,23,0.130,0.174
Bolton Wanderers,1,3,33,0.030,0.091
Bournemouth,0,0,9,0.000,0.000
Bradford City,0,0,2,0.000,0.000


=== By City ===


,top4_by_pos,top4_by_pts,seasons_in_league,top4_by_pos_pct,top4_by_pts_pct
city,,,,,
Barnsley,0,0,1,0.000,0.000
Birmingham,7,7,98,0.071,0.071
Blackburn,3,3,28,0.107,0.107
Blackpool,3,4,23,0.130,0.174
Bolton,1,3,33,0.030,0.091
Bournemouth,0,0,9,0.000,0.000
Bradford,0,0,2,0.000,0.000
Brighton,0,0,13,0.000,0.000
Bristol,0,0,4,0.000,0.000


In [64]:
city_counts.sort_values("top4_by_pts", ascending = False)[:5]

,top4_by_pos,top4_by_pts,seasons_in_league,top4_by_pos_pct,top4_by_pts_pct
city,,,,,
London,88,93,411,0.214,0.226
Manchester,72,73,145,0.497,0.503
Liverpool,59,59,149,0.396,0.396
Leeds,15,15,41,0.366,0.366
Wolverhampton,11,11,46,0.239,0.239


In [71]:
ITALY_CITY_MAP = {
    "Alessandria": "Alessandria",
    "Ancona": "Ancona",
    "Ascoli": "Ascoli Piceno",
    "Atalanta": "Bergamo",
    "Avellino": "Avellino",
    "Bari": "Bari",
    "Benevento": "Benevento",
    "Bologna": "Bologna",
    "Brescia": "Brescia",
    "Cagliari": "Cagliari",
    "Carpi": "Carpi",
    "Catania": "Catania",
    "Catanzaro": "Catanzaro",
    "Cesena": "Cesena",
    "Chievo": "Verona",
    "Como": "Como",
    "Cremonese": "Cremona",
    "Crotone": "Crotone",
    "Empoli": "Empoli",
    "Fiorentina": "Firenze",
    "Foggia": "Foggia",
    "Frosinone": "Frosinone",
    "Genoa": "Genova",
    "Hellas Verona": "Verona",
    "Inter Milan": "Milano",
    "Juventus": "Torino",
    "Lazio": "Roma",
    "Lecce": "Lecce",
    "Lecco": "Lecco",
    "Legnano": "Legnano",
    "Livorno": "Livorno",
    "Lucchese": "Lucca",
    "Mantova": "Mantova",
    "Messina": "Messina",
    "Milan": "Milano",
    "Modena": "Modena",
    "Monza": "Monza",
    "Napoli": "Napoli",
    "Novara": "Novara",
    "Padova": "Padova",
    "Palermo": "Palermo",
    "Parma": "Parma",
    "Perugia": "Perugia",
    "Pescara": "Pescara",
    "Piacenza": "Piacenza",
    "Pisa": "Pisa",
    "Pistoiese": "Pistoia",
    "Pro Patria": "Busto Arsizio",
    "Reggiana": "Reggio Emilia",
    "Reggina": "Reggio Calabria",
    "Roma": "Roma",
    "SPAL": "Ferrara",
    "Salernitana": "Salerno",
    "Sampdoria": "Genova",
    "Sassuolo": "Sassuolo",
    "Siena": "Siena",
    "Spezia": "La Spezia",
    "Ternana": "Terni",
    "Torino": "Torino",
    "Treviso": "Treviso",
    "Triestina": "Trieste",
    "Udinese": "Udine",
    "Varese": "Varese",
    "Venezia": "Venezia",
    "Vicenza": "Vicenza",
}

df_italy["City"] = df_italy["Team"].map(ITALY_CITY_MAP)

unmapped = df_italy[df_italy["City"].isna()]["Team"].unique()
if len(unmapped):
    print(f"Unmapped teams: {unmapped}")


In [79]:
# GD and GAv recomputed from GF and GA
df_italy["GD"] = df_italy["GF"] - df_italy["GA"]
df_italy["GAv"] = (df_italy["GF"] / df_italy["GA"]).round(3)


In [80]:
team_counts.sort_values("top4_by_pts", ascending = False)[:5]

,top4_by_pos,top4_by_pts,seasons_in_league,top4_by_pos_pct,top4_by_pts_pct
team,,,,,
Manchester United,51,52,79,0.646,0.658
Liverpool,45,45,72,0.625,0.625
Arsenal,39,43,80,0.488,0.538
Chelsea,23,24,71,0.324,0.338
Tottenham Hotspur,23,23,75,0.307,0.307


In [81]:
team_counts_ita, city_counts_ita = compute_top4_counts(df_italy)
print("=== By Team ===")
display(team_counts_ita.head())
print("=== By City ===")
display(city_counts_ita.head())


=== By Team ===


,top4_by_pos,top4_by_pts,seasons_in_league,top4_by_pos_pct,top4_by_pts_pct
team,,,,,
Alessandria,0,0,5,0.000,0.000
Ancona,0,0,2,0.000,0.000
Ascoli,1,0,16,0.062,0.000
Atalanta,6,6,61,0.098,0.098
Avellino,0,0,10,0.000,0.000


=== By City ===


,top4_by_pos,top4_by_pts,seasons_in_league,top4_by_pos_pct,top4_by_pts_pct
city,,,,,
Alessandria,0,0,5,0.000,0.0
Ancona,0,0,2,0.000,0.0
Ascoli Piceno,1,0,16,0.062,0.0
Avellino,0,0,10,0.000,0.0
Bari,0,0,21,0.000,0.0


In [82]:
team_counts_ita.sort_values("top4_by_pos", ascending = False)[:5]

,top4_by_pos,top4_by_pts,seasons_in_league,top4_by_pos_pct,top4_by_pts_pct
team,,,,,
Juventus,64,65,79,0.810,0.823
Inter Milan,56,58,80,0.700,0.725
Milan,53,54,78,0.679,0.692
Napoli,27,27,67,0.403,0.403
Fiorentina,25,26,77,0.325,0.338


In [83]:
team_counts.sort_values("top4_by_pos", ascending = False)[:5]

,top4_by_pos,top4_by_pts,seasons_in_league,top4_by_pos_pct,top4_by_pts_pct
team,,,,,
Manchester United,51,52,79,0.646,0.658
Liverpool,45,45,72,0.625,0.625
Arsenal,39,43,80,0.488,0.538
Chelsea,23,24,71,0.324,0.338
Tottenham Hotspur,23,23,75,0.307,0.307


In [2]:
def compute_league_winners(df: pd.DataFrame) -> pd.DataFrame:
    return df[df['Pos'] == 1].reset_index(drop = True)

In [86]:
italy_winners = compute_league_winners(df_italy)
england_winners = compute_league_winners(df_england)

In [88]:
italy_winners['Team'].value_counts()[:5]

Team
Juventus       30
Milan          16
Inter Milan    16
Torino          4
Napoli          4
Name: count, dtype: int64

In [89]:
england_winners['Team'].value_counts()[:5]

Team
Manchester United    18
Liverpool            16
Arsenal               9
Manchester City       9
Chelsea               6
Name: count, dtype: int64

In [92]:
england_winners['City'].value_counts(normalize=True)[:5]

City
Manchester       0.3375
Liverpool        0.2500
London           0.2125
Wolverhampton    0.0375
Leeds            0.0375
Name: proportion, dtype: float64

In [93]:
italy_winners['City'].value_counts(normalize=True)[:5]

City
Torino     0.425
Milano     0.400
Roma       0.050
Napoli     0.050
Firenze    0.025
Name: proportion, dtype: float64

In [94]:
italy_winners['Team'].value_counts(normalize=True)[:5]

Team
Juventus       0.375
Milan          0.200
Inter Milan    0.200
Torino         0.050
Napoli         0.050
Name: proportion, dtype: float64

In [95]:
england_winners['Team'].value_counts(normalize = True)[:5]

Team
Manchester United    0.2250
Liverpool            0.2000
Arsenal              0.1125
Manchester City      0.1125
Chelsea              0.0750
Name: proportion, dtype: float64

## Function: `compute_dominant_teams`

Given a list of teams and a league dataframe, returns a season-by-season summary of how
many of those teams finished in the top N positions, along with their individual positions and points.

### Parameters

| Parameter | Default | Description |
|---|---|---|
| `df` | required | League standings dataframe |
| `teams` | required | Ordered list of teams to track. The order determines the order of values in `positions` and `points` columns |
| `label` | `"dominant"` | Label used to name the output columns: `number_of_{label}` and `{label}` |
| `top_n` | `4` | Number of top positions to consider |
| `definition` | `"pos"` | `"pos"` uses the official final position; `"pts"` uses points rank (ties share rank, may include more than `top_n` teams) |
| `pos_col` | `"Pos"` | Name of the position column in the dataframe |
| `pts_col` | `"Pts"` | Name of the points column in the dataframe |

### Output columns

| Column | Description |
|---|---|
| `season` | Season label (e.g. `1946/47`) |
| `number_of_{label}` | How many of the tracked teams finished in the top N that season |
| `{label}` | List of tracked teams that finished in the top N, in the order defined by `teams` |
| `positions` | Official position (or points rank) of each tracked team, in the order defined by `teams`. `-1` if the team was not in the league that season |
| `points` | Points of each tracked team, in the order defined by `teams`. `-1` if the team was not in the league that season |

### Notes

- The order of `teams` is preserved in `positions` and `points` — this makes downstream
  plotting and indexing predictable.
- Teams absent from the league in a given season are assigned `-1` for both position and
  points, rather than `NaN`, to keep the columns as integer lists suitable for visualization.
- When `definition="pts"`, ties on points mean more than `top_n` teams may qualify in a
  given season. This is documented as a known limitation and will be revisited in a later stage.


In [97]:
def compute_dominant_teams(
    df: pd.DataFrame,
    teams: list[str],
    label: str = "dominant",
    top_n: int = 4,
    definition: str = "pos",  # "pos" or "pts"
    pos_col: str = "Pos",
    pts_col: str = "Pts",
) -> pd.DataFrame:
    
    def parse_val(val):
        if pd.isna(val):
            return pd.NA
        return int(re.sub(r'\[.*?\]', '', str(val)).strip())

    df = df.copy()
    df["_pos"] = df[pos_col].apply(parse_val)
    df["_pts"] = df[pts_col].apply(parse_val)

    records = []
    for season, group in df.groupby("season"):

        if definition == "pos":
            top_n_threshold = top_n
            in_top = lambda row: row["_pos"] <= top_n_threshold
        else:
            nth_pts = group["_pts"].nlargest(top_n).min()
            in_top = lambda row: row["_pts"] >= nth_pts

        top_n_teams = set(group[group.apply(in_top, axis=1)]["Team"])
        teams_in_top = [t for t in teams if t in top_n_teams]

        row_data = {
            "season": season,
            f"number_of_{label}": len(teams_in_top),
            label: teams_in_top,
        }

        for team in teams:
            team_row = group[group["Team"] == team]
            if team_row.empty:
                row_data[f"pos_{team}"] = -1
                row_data[f"pts_{team}"] = -1
            else:
                row_data[f"pos_{team}"] = int(team_row["_pos"].values[0])
                row_data[f"pts_{team}"] = int(team_row["_pts"].values[0])

        records.append(row_data)

    result = pd.DataFrame(records)
    result["positions"] = result[[f"pos_{t}" for t in teams]].values.tolist()
    result["points"]    = result[[f"pts_{t}" for t in teams]].values.tolist()

    return result[["season", f"number_of_{label}", label, "positions", "points"]]



In [98]:
# Example for Italy
STRISCIATE = ["Juventus", "Inter Milan", "Milan"]
strisciate_df = compute_dominant_teams(
    df_italy,
    teams=STRISCIATE,
    label="strisciate",
    top_n=4,
    definition="pos",
)
strisciate_df.head()


,season,number_of_strisciate,strisciate,positions,points
0,1946/47,2,"[Juventus, Milan]","[2, 10, 4]","[53, 36, 50]"
1,1947/48,2,"[Juventus, Milan]","[3, 12, 2]","[49, 37, 49]"
2,1948/49,3,"[Juventus, Inter Milan, Milan]","[4, 2, 3]","[44, 55, 50]"
3,1949/50,3,"[Juventus, Inter Milan, Milan]","[1, 3, 2]","[62, 49, 57]"
4,1950/51,3,"[Juventus, Inter Milan, Milan]","[3, 2, 1]","[54, 59, 60]"


In [ ]:
strisciate_df.value_counts("number_of_strisciate").sort_index(ascending=False).reset_index().rename(columns={"number_of_strisciate": "numero di strisciate", "count": "totale"})


,numero di strisciate,totale
0,3,28
1,2,37
2,1,15


In [109]:
# Example for England
# Manchester United    0.2250
# Liverpool            0.2000
# Arsenal              0.1125
# Manchester City      0.1125
# Chelsea              0.0750

DOMINANT_ENG = ["Manchester United", "Liverpool", "Manchester City"]
dominant_eng_df = compute_dominant_teams(
    df_england,
    teams=DOMINANT_ENG,
    label="dominant_eng",
    top_n=4,
    definition="pos",
)

dominant_eng_df.head()

,season,number_of_dominant_eng,dominant_eng,positions,points
0,1946/47,2,"[Manchester United, Liverpool]","[2, 1, -1]","[56, 57, -1]"
1,1947/48,1,[Manchester United],"[2, 11, 10]","[52, 42, 42]"
2,1948/49,1,[Manchester United],"[2, 12, 7]","[53, 40, 45]"
3,1949/50,1,[Manchester United],"[4, 8, 21]","[50, 48, 29]"
4,1950/51,1,[Manchester United],"[2, 9, -1]","[56, 43, -1]"


In [110]:
dominant_eng_df.value_counts("number_of_dominant_eng").sort_index(ascending=False).reset_index().rename(columns={"number_of_dominant_eng": "numero di eng teams", "count": "totale"})

,numero di eng teams,totale
0,3,4
1,2,39
2,1,27
3,0,10


In [111]:
43/80

0.5375

### Diagnostic Functions - who are the other Leagues Strisciate?

In [114]:
from itertools import combinations

def compute_dominant_combinations(
    df: pd.DataFrame,
    teams: list[str],
    combo_size: int = 3,
    top_n: int = 4,
    definition: str = "pos",
    pos_col: str = "Pos",
    pts_col: str = "Pts",
) -> pd.DataFrame:

    def parse_val(val):
        if pd.isna(val):
            return pd.NA
        return int(re.sub(r'\[.*?\]', '', str(val)).strip())

    df = df.copy()
    df["_pos"] = df[pos_col].apply(parse_val)
    df["_pts"] = df[pts_col].apply(parse_val)

    # Pre-compute top_n sets per season
    season_top = {}
    for season, group in df.groupby("season"):
        if definition == "pos":
            season_top[season] = set(group[group["_pos"] <= top_n]["Team"])
        else:
            nth_pts = group["_pts"].nlargest(top_n).min()
            season_top[season] = set(group[group["_pts"] >= nth_pts]["Team"])

    total_seasons = len(season_top)
    records = []

    for combo in combinations(teams, combo_size):
        counts = {i: 0 for i in range(combo_size + 1)}

        for season, top_set in season_top.items():
            n_in_top = sum(t in top_set for t in combo)
            counts[n_in_top] += 1

        row = {"combination": combo}
        for i in range(combo_size + 1):
            row[f"{i}_of_{combo_size}"]     = counts[i]
            row[f"{i}_of_{combo_size}_pct"] = round(counts[i] / total_seasons, 3)

        # at_least_{combo_size-1}_of_{combo_size}
        threshold = combo_size - 1
        row[f"at_least_{threshold}_of_{combo_size}"] = sum(
            counts[i] for i in range(threshold, combo_size + 1)
        )
        row[f"at_least_{threshold}_of_{combo_size}_pct"] = round(
            row[f"at_least_{threshold}_of_{combo_size}"] / total_seasons, 3
        )

        records.append(row)

    result = pd.DataFrame(records).sort_values(
        f"{combo_size}_of_{combo_size}_pct", ascending=False
    ).reset_index(drop=True)

    return result

In [118]:
# Example for England — find the Italian-like triplet
BIG_CANDIDATES = ["Manchester United", "Liverpool", "Arsenal", "Manchester City", "Chelsea"]

combo_df = compute_dominant_combinations(
    df_england,
    teams=BIG_CANDIDATES,
    combo_size=3,
    top_n=4,
    definition="pos",
)
combo_df.sort_values("at_least_2_of_3", ascending = False)[:5]


,combination,0_of_3,0_of_3_pct,1_of_3,1_of_3_pct,2_of_3,2_of_3_pct,3_of_3,3_of_3_pct,at_least_2_of_3,at_least_2_of_3_pct
1,"(Manchester United, Liverpool, Arsenal)",7,0.087,22,0.275,40,0.500,11,0.138,51,0.637
8,"(Manchester United, Liverpool, Manchester City)",10,0.125,27,0.338,39,0.487,4,0.050,43,0.537
3,"(Manchester United, Liverpool, Chelsea)",9,0.113,31,0.388,32,0.400,8,0.100,40,0.500
9,"(Liverpool, Arsenal, Manchester City)",15,0.188,28,0.350,34,0.425,3,0.037,37,0.463
5,"(Manchester United, Arsenal, Manchester City)",10,0.125,35,0.438,29,0.362,6,0.075,35,0.438


In [119]:
# Save Italy
italy_out = Path("data/processed")
italy_out.mkdir(parents=True, exist_ok=True)
df_italy.to_csv(italy_out / "serie_a_final_standings.csv", index=False)
print(f"Italy saved: {len(df_italy)} rows, {df_italy['season'].nunique()} seasons")

# Save England
df_england.to_csv(italy_out / "england_top_league_standings.csv", index=False)
print(f"England saved: {len(df_england)} rows, {df_england['season'].nunique()} seasons")


Italy saved: 1455 rows, 80 seasons
England saved: 1691 rows, 80 seasons


## Scraping German Data

In [120]:
test_seasons_germany = {
    "Early Bundesliga": 1965,
    "Mid era": 1980,
    "Around 2000": 1999,
    "Modern": 2010,
    "Recent": 2022,
}

for label, year in test_seasons_germany.items():
    season_end = year + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    url = f"https://en.wikipedia.org/wiki/{year}%E2%80%93{suffix}_Fu%C3%9Fball-Bundesliga"
    print(f"\n{'='*60}")
    print(f"{label} — {year}: {url}")
    response = requests.get(url, headers=HEADERS, timeout=15)
    print(f"  Status: {response.status_code}")
    if response.status_code == 200:
        tables = pd.read_html(io.StringIO(response.text))
        print(f"  {len(tables)} tables found")
        for i, t in enumerate(tables[:6]):
            print(f"  Table {i}: shape={t.shape}, columns={list(t.columns[:6])}...")
    time.sleep(1.5)



Early Bundesliga — 1965: https://en.wikipedia.org/wiki/1965%E2%80%9366_Fu%C3%9Fball-Bundesliga
  Status: 200
  7 tables found
  Table 0: shape=(13, 2), columns=[0, 1]...
  Table 1: shape=(18, 3), columns=['Club', 'Ground[5]', 'Capacity[5]']...
  Table 2: shape=(18, 11), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 3: shape=(18, 19), columns=['Home \\ Away', 'SCT', 'EBS', 'SVW', 'BVB', 'SGE']...
  Table 4: shape=(1, 1), columns=['TSV 1860 München']...
  Table 5: shape=(7, 2), columns=['vteBundesliga', 'vteBundesliga.1']...

Mid era — 1980: https://en.wikipedia.org/wiki/1980%E2%80%9381_Fu%C3%9Fball-Bundesliga
  Status: 200
  8 tables found
  Table 0: shape=(14, 2), columns=[0, 1]...
  Table 1: shape=(18, 4), columns=['Club', 'Location', 'Ground[3]', 'Capacity[3]']...
  Table 2: shape=(18, 11), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 3: shape=(18, 19), columns=['Home \\ Away', 'DSC', 'BOC', 'BVB', 'DUI', 'F95']...
  Table 4: shape=(1, 1), columns=['FC Bayern 

In [121]:
for year in range(2018, 2024):
    season_end = year + 1
    suffix = str(season_end)[-2:]
    for page_name in ["Fu%C3%9Fball-Bundesliga", "Bundesliga"]:
        url = f"https://en.wikipedia.org/wiki/{year}%E2%80%93{suffix}_{page_name}"
        response = requests.get(url, headers=HEADERS, timeout=15)
        print(f"{year}: {page_name} → {response.status_code}")
    time.sleep(1.5)


2018: Fu%C3%9Fball-Bundesliga → 404
2018: Bundesliga → 200
2019: Fu%C3%9Fball-Bundesliga → 404
2019: Bundesliga → 200
2020: Fu%C3%9Fball-Bundesliga → 404
2020: Bundesliga → 200
2021: Fu%C3%9Fball-Bundesliga → 404
2021: Bundesliga → 200
2022: Fu%C3%9Fball-Bundesliga → 404
2022: Bundesliga → 200
2023: Fu%C3%9Fball-Bundesliga → 404
2023: Bundesliga → 200


In [122]:
for year in range(2010, 2019):
    season_end = year + 1
    suffix = str(season_end)[-2:]
    for page_name in ["Fu%C3%9Fball-Bundesliga", "Bundesliga"]:
        url = f"https://en.wikipedia.org/wiki/{year}%E2%80%93{suffix}_{page_name}"
        response = requests.get(url, headers=HEADERS, timeout=15)
        print(f"{year}: {page_name} → {response.status_code}")
    time.sleep(1.5)


2010: Fu%C3%9Fball-Bundesliga → 200
2010: Bundesliga → 200
2011: Fu%C3%9Fball-Bundesliga → 200
2011: Bundesliga → 200
2012: Fu%C3%9Fball-Bundesliga → 200
2012: Bundesliga → 200
2013: Fu%C3%9Fball-Bundesliga → 200
2013: Bundesliga → 200
2014: Fu%C3%9Fball-Bundesliga → 200
2014: Bundesliga → 200
2015: Fu%C3%9Fball-Bundesliga → 404
2015: Bundesliga → 200
2016: Fu%C3%9Fball-Bundesliga → 404
2016: Bundesliga → 200
2017: Fu%C3%9Fball-Bundesliga → 404
2017: Bundesliga → 200
2018: Fu%C3%9Fball-Bundesliga → 404
2018: Bundesliga → 200


In [123]:
def build_url_germany(season_start: int) -> str:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    if season_start < 2015:
        page = f"{season_start}%E2%80%93{suffix}_Fu%C3%9Fball-Bundesliga"
    else:
        page = f"{season_start}%E2%80%93{suffix}_Bundesliga"
    return f"https://en.wikipedia.org/wiki/{page}"

test_seasons_germany = {
    "Early": 1965,
    "Mid": 1980,
    "Around 2000": 1999,
    "Pre-cutoff": 2013,
    "Post-cutoff": 2016,
    "Recent": 2023,
}

for label, year in test_seasons_germany.items():
    url = build_url_germany(year)
    print(f"\n{label} — {year}")
    try:
        t = get_england_table(year)  # reuse same logic, just pass url manually
        response = requests.get(url, headers=HEADERS, timeout=15)
        tables = pd.read_html(io.StringIO(response.text))
        standings = find_standings_table(tables, str(year))
        if standings is not None:
            print(standings[["Pos", "Team", "Pld", "Pts"]].head(3).to_string(index=False))
        else:
            print("  no standings table found")
    except Exception as e:
        print(f"  ERROR: {e}")
    time.sleep(1.5)



Early — 1965
 Pos              Team  Pld  Pts
   1   1860 Munich (C)   34   50
   2 Borussia Dortmund   34   47
   3     Bayern Munich   34   47

Mid — 1980
 Pos              Team  Pld  Pts
   1 Bayern Munich (C)   34   53
   2      Hamburger SV   34   49
   3     VfB Stuttgart   34   46

Around 2000 — 1999
 Pos              Team  Pld Pts
   1 Bayern Munich (C)   34  73
   2  Bayer Leverkusen   34  73
   3      Hamburger SV   34  59

Pre-cutoff — 2013
 Pos              Team  Pld  Pts
   1 Bayern Munich (C)   34   90
   2 Borussia Dortmund   34   71
   3        Schalke 04   34   64

Post-cutoff — 2016
 Pos              Team  Pld  Pts
   1 Bayern Munich (C)   34   82
   2        RB Leipzig   34   67
   3 Borussia Dortmund   34   64

Recent — 2023
 Pos                 Team  Pld  Pts
   1 Bayer Leverkusen (C)   34   90
   2        VfB Stuttgart   34   73
   3        Bayern Munich   34   72


In [124]:
def get_germany_table(season_start: int) -> pd.DataFrame | None:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    season_label = f"{season_start}/{suffix}"
    url = build_url_germany(season_start)

    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    try:
        tables = pd.read_html(io.StringIO(response.text))
    except Exception as e:
        raise RuntimeError(f"Failed to parse tables from {url}: {e}") from e

    return find_standings_table(tables, season_label)


def scrape_germany_seasons(
    start: int = 1963,
    end: int = 2024,
    output_dir: str = "data/processed",
    sleep_seconds: float = 1.5,
) -> pd.DataFrame:
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    all_tables = []
    failed = []

    for year in range(start, end + 1):
        try:
            print(f"{year}: scraping...", end=" ", flush=True)
            t = get_germany_table(year)
            if t is not None:
                all_tables.append(t)
                print(f"OK — {len(t)} rows, columns: {list(t.columns)}")
            else:
                print("no standings table found")
                failed.append(year)
        except Exception as e:
            print(f"ERROR — {e}")
            failed.append(year)

        time.sleep(sleep_seconds)

    if not all_tables:
        print("No data collected.")
        return pd.DataFrame()

    df = pd.concat(all_tables, ignore_index=True)

    csv_path = out_path / "germany_bundesliga_standings.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved {len(df)} rows to {csv_path}")

    if failed:
        print(f"Failed seasons: {failed}")

    return df


# Run
df_germany = scrape_germany_seasons(start=1963, end=2024)
df_germany.head()


1963: scraping... OK — 16 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GR', 'Pts', 'Qualification or relegation', 'season']
1964: scraping... OK — 16 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GR', 'Pts', 'Qualification or relegation', 'season']
1965: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GR', 'Pts', 'Qualification or relegation', 'season']
1966: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GR', 'Pts', 'Qualification or relegation', 'season']
1967: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GR', 'Pts', 'Qualification or relegation', 'season']
1968: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GR', 'Pts', 'Qualification or relegation', 'season']
1969: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Qualification or relegati

,Pos,Team,Pld,W,D,L,GF,GA,GR,Pts,Qualification or relegation,season,GD
0,1,1. FC Köln (C),30,17,11,2,78,40,1.950,45,Qualification to European Cup preliminary round,1963/64,NaN
1,2,Meidericher SV,30,13,13,4,60,36,1.667,39,NaN,1963/64,NaN
2,3,Eintracht Frankfurt,30,16,7,7,65,41,1.585,39,Qualification to Inter-Cities Fairs Cup first ...,1963/64,NaN
3,4,Borussia Dortmund,30,14,5,11,73,57,1.281,33,Qualification to Inter-Cities Fairs Cup first ...,1963/64,NaN
4,5,VfB Stuttgart,30,13,7,10,48,40,1.200,33,Qualification to Inter-Cities Fairs Cup first ...,1963/64,NaN


In [125]:
df_germany.isna().sum()

Pos                               0
Team                              0
Pld                               0
W                                 0
D                                 0
L                                 0
GF                                0
GA                                0
GR                             1010
Pts                               0
Qualification or relegation     503
season                            0
GD                              104
dtype: int64

In [126]:
df_germany[df_germany['GD'].isna()]['season'].unique()

<StringArray>
['1963/64', '1964/65', '1965/66', '1966/67', '1967/68', '1968/69']
Length: 6, dtype: str

In [127]:
df_germany[df_germany['GR'].isna()]['season'].unique()

<StringArray>
[  '1969/70',   '1970/71',   '1971/72',   '1972/73',   '1973/74',   '1974/75',
   '1975/76',   '1976/77',   '1977/78',   '1978/79',   '1979/80',   '1980/81',
   '1981/82',   '1982/83',   '1983/84',   '1984/85',   '1985/86',   '1986/87',
   '1987/88',   '1988/89',   '1989/90',   '1990/91',   '1991/92',   '1992/93',
   '1993/94',   '1994/95',   '1995/96',   '1996/97',   '1997/98',   '1998/99',
 '1999/2000',   '2000/01',   '2001/02',   '2002/03',   '2003/04',   '2004/05',
   '2005/06',   '2006/07',   '2007/08',   '2008/09',   '2009/10',   '2010/11',
   '2011/12',   '2012/13',   '2013/14',   '2014/15',   '2015/16',   '2016/17',
   '2017/18',   '2018/19',   '2019/20',   '2020/21',   '2021/22',   '2022/23',
   '2023/24',   '2024/25']
Length: 56, dtype: str

In [128]:
df_germany['Team'].nunique()

140

In [129]:
np.sort(df_germany['Team'].unique())

array(['1. FC Heidenheim', '1. FC Heidenheim (O)', '1. FC Kaiserslautern',
       '1. FC Kaiserslautern (C)', '1. FC Kaiserslautern (R)',
       '1. FC Kaiserslautern[b]', '1. FC Kaiserslautern[b] (R)',
       '1. FC Köln', '1. FC Köln (C)', '1. FC Köln (O)', '1. FC Köln (R)',
       '1. FC Nürnberg', '1. FC Nürnberg (C)', '1. FC Nürnberg (O)',
       '1. FC Nürnberg (R)', '1. FC Saarbrücken', '1. FC Saarbrücken (R)',
       '1860 Munich', '1860 Munich (C)', '1860 Munich (R)',
       '1899 Hoffenheim', '1899 Hoffenheim (O)', 'Alemannia Aachen',
       'Alemannia Aachen (R)', 'Arminia Bielefeld',
       'Arminia Bielefeld (R)', 'Arminia Bielefeld[c] (R)',
       'Bayer 05 Uerdingen', 'Bayer 05 Uerdingen (R)', 'Bayer Leverkusen',
       'Bayer Leverkusen (C)', 'Bayer Leverkusen (O)', 'Bayern Munich',
       'Bayern Munich (C)', 'Blau-Weiß 90 Berlin (R)',
       'Borussia Dortmund', 'Borussia Dortmund (C)',
       'Borussia Dortmund (O)', 'Borussia Dortmund (R)',
       'Borussia Möncheng

1. In 1953, the club merged with the Werkssportgruppen Bayer AG Uerdingen, the local worker's sports club of the chemical giant Bayer AG, becoming FC Bayer 05 Uerdingen. Bayer withdrew its sponsorship of the football team in 1995 at which time the club took on the name Krefelder Fußball-Club Uerdingen 05. Bayer continues to support the non-footballing departments of the club as Sport-Club Bayer 05 Uerdingen.

2. In 1966, they finished with a club record 70 goals scored in the league, including the Bundesliga's biggest ever away win, 9–0 against Tasmania Berlin. They also reached the DFB-Pokal final, losing to Bayern Munich.[1][9] This was the last season played under the old name of Meidericher SV, as the club renamed itself MSV Duisburg in 1967, having received financial support from the city of Duisburg.[10]

In [130]:
df_england.columns

Index(['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts',
       'Relegation', 'season', 'Qualification or relegation', 'GD', 'City'],
      dtype='str')

- Germany cleaning functions

In [132]:
GERMANY_NAME_MAP = {
    # Encoding fixes
    "Borussia\xa0Mönchengladbach": "Borussia Mönchengladbach",
    "Hansa\xa0Rostock": "Hansa Rostock",
    # Rebrands
    "FSV Mainz 05": "Mainz 05",
    "1899 Hoffenheim": "TSG Hoffenheim",
    "Bayer 05 Uerdingen": "KFC Uerdingen",
    "Meidericher SV": "MSV Duisburg",
}

GERMANY_CITY_MAP = {
    "1. FC Heidenheim": "Heidenheim an der Brenz",
    "1. FC Kaiserslautern": "Kaiserslautern",
    "1. FC Köln": "Köln",
    "1. FC Nürnberg": "Nürnberg",
    "1. FC Saarbrücken": "Saarbrücken",
    "1860 Munich": "München",
    "1899 Hoffenheim": "Sinsheim",
    "Alemannia Aachen": "Aachen",
    "Arminia Bielefeld": "Bielefeld",
    "Bayer Leverkusen": "Leverkusen",
    "Bayern Munich": "München",
    "Blau-Weiß 90 Berlin": "Berlin",
    "Borussia Dortmund": "Dortmund",
    "Borussia Mönchengladbach": "Mönchengladbach",
    "Borussia Neunkirchen": "Neunkirchen",
    "Darmstadt 98": "Darmstadt",
    "Dynamo Dresden": "Dresden",
    "Eintracht Braunschweig": "Braunschweig",
    "Eintracht Frankfurt": "Frankfurt",
    "Energie Cottbus": "Cottbus",
    "FC 08 Homburg": "Homburg",
    "FC Augsburg": "Augsburg",
    "FC Ingolstadt": "Ingolstadt",
    "FC St. Pauli": "Hamburg",
    "Fortuna Düsseldorf": "Düsseldorf",
    "Fortuna Köln": "Köln",
    "Greuther Fürth": "Fürth",
    "Hamburger SV": "Hamburg",
    "Hannover 96": "Hannover",
    "Hansa Rostock": "Rostock",
    "Hertha BSC": "Berlin",
    "Holstein Kiel": "Kiel",
    "KFC Uerdingen": "Krefeld",
    "Karlsruher SC": "Karlsruhe",
    "Kickers Offenbach": "Offenbach am Main",
    "MSV Duisburg": "Duisburg",
    "Mainz 05": "Mainz",
    "Preußen Münster": "Münster",
    "RB Leipzig": "Leipzig",
    "Rot-Weiss Essen": "Essen",
    "Rot-Weiß Oberhausen": "Oberhausen",
    "SC Freiburg": "Freiburg im Breisgau",
    "SC Paderborn": "Paderborn",
    "SG Wattenscheid 09": "Bochum",
    "SSV Ulm 1846": "Ulm",
    "SV Darmstadt 98": "Darmstadt",
    "Schalke 04": "Gelsenkirchen",
    "SpVgg Unterhaching": "Unterhaching",
    "Stuttgarter Kickers": "Stuttgart",
    "TSG Hoffenheim": "Sinsheim",
    "Tasmania Berlin": "Berlin",
    "Tennis Borussia Berlin": "Berlin",
    "Union Berlin": "Berlin",
    "VfB Leipzig": "Leipzig",
    "VfB Stuttgart": "Stuttgart",
    "VfL Bochum": "Bochum",
    "VfL Wolfsburg": "Wolfsburg",
    "Waldhof Mannheim": "Mannheim",
    "Werder Bremen": "Bremen",
    "Wuppertaler SV": "Wuppertal",
}

def clean_germany_team(name: str) -> str:
    if pd.isna(name):
        return name
    # Apply name map before stripping (catches encoding issues)
    name = GERMANY_NAME_MAP.get(name, name)
    name = re.sub(r'\[.*?\]', '', name)
    name = re.sub(r'\(.*?\)', '', name)
    name = name.strip()
    # Apply name map again after stripping (catches rebrands without suffixes)
    return GERMANY_NAME_MAP.get(name, name)

df_germany["Team"] = df_germany["Team"].apply(clean_germany_team)
df_germany["GD"] = df_germany["GF"] - df_germany["GA"]
df_germany["GAv"] = (df_germany["GF"] / df_germany["GA"]).round(3)
df_germany["City"] = df_germany["Team"].map(GERMANY_CITY_MAP)

unmapped = df_germany[df_germany["City"].isna()]["Team"].unique()
if len(unmapped):
    print(f"Unmapped teams: {unmapped}")

df_germany.to_csv(Path("data/processed") / "germany_bundesliga_standings.csv", index=False)
print(f"Germany saved: {len(df_germany)} rows, {df_germany['season'].nunique()} seasons")


Germany saved: 1114 rows, 62 seasons


In [134]:
df_germany['season'].nunique()

62

In [142]:
compute_league_winners(df_germany)['Team'].value_counts()

Team
Bayern Munich               33
Borussia Mönchengladbach     5
Borussia Dortmund            5
Werder Bremen                4
Hamburger SV                 3
VfB Stuttgart                3
1. FC Köln                   2
1. FC Kaiserslautern         2
1860 Munich                  1
Eintracht Braunschweig       1
1. FC Nürnberg               1
VfL Wolfsburg                1
Bayer Leverkusen             1
Name: count, dtype: int64

In [145]:
g1 = compute_league_winners(df_germany)
g1[g1['Team'] == '1. FC Nürnberg']

,Pos,Team,Pld,W,D,L,GF,GA,GR,Pts,Qualification or relegation,season,GD,GAv,City
4,1,1. FC Nürnberg,34,19,9,6,71,37,1.919,47,Qualification to European Cup first round,1967/68,34,1.919,Nürnberg


In [140]:
df_germany['season'].unique()

<StringArray>
[  '1963/64',   '1964/65',   '1965/66',   '1966/67',   '1967/68',   '1968/69',
   '1969/70',   '1970/71',   '1971/72',   '1972/73',   '1973/74',   '1974/75',
   '1975/76',   '1976/77',   '1977/78',   '1978/79',   '1979/80',   '1980/81',
   '1981/82',   '1982/83',   '1983/84',   '1984/85',   '1985/86',   '1986/87',
   '1987/88',   '1988/89',   '1989/90',   '1990/91',   '1991/92',   '1992/93',
   '1993/94',   '1994/95',   '1995/96',   '1996/97',   '1997/98',   '1998/99',
 '1999/2000',   '2000/01',   '2001/02',   '2002/03',   '2003/04',   '2004/05',
   '2005/06',   '2006/07',   '2007/08',   '2008/09',   '2009/10',   '2010/11',
   '2011/12',   '2012/13',   '2013/14',   '2014/15',   '2015/16',   '2016/17',
   '2017/18',   '2018/19',   '2019/20',   '2020/21',   '2021/22',   '2022/23',
   '2023/24',   '2024/25']
Length: 62, dtype: str

In [146]:
df_germany.columns

Index(['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GR', 'Pts',
       'Qualification or relegation', 'season', 'GD', 'GAv', 'City'],
      dtype='str')

In [147]:
df_england.columns

Index(['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts',
       'Relegation', 'season', 'Qualification or relegation', 'GD', 'City'],
      dtype='str')

## Scraping Spain data

In [148]:
def build_url_spain(season_start: int) -> str:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    return f"https://en.wikipedia.org/wiki/{season_start}%E2%80%93{suffix}_La_Liga"

test_seasons_spain = {
    "Early": 1950,
    "Mid": 1975,
    "Around 2000": 1999,
    "Modern": 2010,
    "Recent": 2023,
}

for label, year in test_seasons_spain.items():
    url = build_url_spain(year)
    print(f"\n{'='*60}")
    print(f"{label} — {year}: {url}")
    response = requests.get(url, headers=HEADERS, timeout=15)
    print(f"  Status: {response.status_code}")
    if response.status_code == 200:
        tables = pd.read_html(io.StringIO(response.text))
        print(f"  {len(tables)} tables found")
        for i, t in enumerate(tables[:6]):
            print(f"  Table {i}: shape={t.shape}, columns={list(t.columns[:6])}...")
    time.sleep(1.5)



Early — 1950: https://en.wikipedia.org/wiki/1950%E2%80%9351_La_Liga
  Status: 200
  11 tables found
  Table 0: shape=(15, 2), columns=[0, 1]...
  Table 1: shape=(16, 3), columns=['Club', 'City', 'Stadium']...
  Table 2: shape=(16, 11), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 3: shape=(16, 17), columns=['Home \\ Away', 'ALC', 'ATB', 'ATM', 'BAR', 'MAL']...
  Table 4: shape=(6, 11), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 5: shape=(6, 7), columns=['Home \\ Away', 'LPA', 'MAL', 'MUR', 'SAB', 'SAL']...

Mid — 1975: https://en.wikipedia.org/wiki/1975%E2%80%9376_La_Liga
  Status: 200
  9 tables found
  Table 0: shape=(10, 2), columns=[0, 1]...
  Table 1: shape=(18, 3), columns=['Team', 'Home city', 'Stadium']...
  Table 2: shape=(18, 11), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 3: shape=(18, 19), columns=['Home \\ Away', 'ATB', 'ATM', 'BAR', 'BET', 'ELC']...
  Table 4: shape=(10, 4), columns=['Rank', 'Player', 'Club', 'Goals']...
  Table 5:

In [149]:
for label, year in test_seasons_spain.items():
    print(f"\n{label} — {year}")
    try:
        url = build_url_spain(year)
        response = requests.get(url, headers=HEADERS, timeout=15)
        tables = pd.read_html(io.StringIO(response.text))
        standings = find_standings_table(tables, str(year))
        if standings is not None:
            print(standings[["Pos", "Team", "Pld", "Pts"]].head(3).to_string(index=False))
            print(f"  {len(standings)} teams")
        else:
            print("  no standings table found")
    except Exception as e:
        print(f"  ERROR: {e}")
    time.sleep(1.5)



Early — 1950
 Pos                Team  Pld  Pts
   1 Atlético Madrid (C)   30   40
   2             Sevilla   30   38
   3            Valencia   30   37
  16 teams

Mid — 1975
 Pos            Team  Pld  Pts
   1 Real Madrid (C)   34   48
   2       Barcelona   34   43
   3 Atlético Madrid   34   42
  18 teams

Around 2000 — 1999
 Pos                    Team  Pld   Pts
   1 Deportivo La Coruña (C)   38    69
   2               Barcelona   38 64[a]
   3                Valencia   38 64[a]
  20 teams

Modern — 2010
 Pos          Team  Pld Pts
   1 Barcelona (C)   38  96
   2   Real Madrid   38  92
   3      Valencia   38  71
  20 teams

Recent — 2023
 Pos            Team  Pld Pts
   1 Real Madrid (C)   38  95
   2       Barcelona   38  85
   3          Girona   38  81
  20 teams


In [150]:
def get_spain_table(season_start: int) -> pd.DataFrame | None:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    season_label = f"{season_start}/{suffix}"
    url = build_url_spain(season_start)

    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    try:
        tables = pd.read_html(io.StringIO(response.text))
    except Exception as e:
        raise RuntimeError(f"Failed to parse tables from {url}: {e}") from e

    return find_standings_table(tables, season_label)


def scrape_spain_seasons(
    start: int = 1946,
    end: int = 2024,
    output_dir: str = "data/processed",
    sleep_seconds: float = 1.5,
) -> pd.DataFrame:
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    all_tables = []
    failed = []

    for year in range(start, end + 1):
        try:
            print(f"{year}: scraping...", end=" ", flush=True)
            t = get_spain_table(year)
            if t is not None:
                all_tables.append(t)
                print(f"OK — {len(t)} rows, columns: {list(t.columns)}")
            else:
                print("no standings table found")
                failed.append(year)
        except Exception as e:
            print(f"ERROR — {e}")
            failed.append(year)

        time.sleep(sleep_seconds)

    if not all_tables:
        print("No data collected.")
        return pd.DataFrame()

    df = pd.concat(all_tables, ignore_index=True)

    csv_path = out_path / "spain_la_liga_standings.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved {len(df)} rows to {csv_path}")

    if failed:
        print(f"Failed seasons: {failed}")

    return df

In [160]:
# Run
df_spain = scrape_spain_seasons(start=1946, end=2025)
df_spain.head()


1946: scraping... OK — 14 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Qualification or relegation', 'season']
1947: scraping... OK — 14 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Relegation', 'season']
1948: scraping... OK — 14 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Qualification or relegation', 'season']
1949: scraping... OK — 14 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Qualification or relegation', 'season']
1950: scraping... OK — 16 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Qualification or relegation', 'season']
1951: scraping... OK — 16 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Qualification or relegation', 'season']
1952: scraping... OK — 16 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Qualification or relegation', 'season']
19

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts,Qualification or relegation,season,Relegation,Qualification
0,1,Valencia (C),26,16,2,8,54,34,+20,34[a],NaN,1946/47,NaN,NaN
1,2,Atlético Bilbao,26,15,4,7,72,38,+34,34[a],NaN,1946/47,NaN,NaN
2,3,Atlético Madrid,26,13,6,7,58,44,+14,32,NaN,1946/47,NaN,NaN
3,4,Barcelona,26,14,3,9,59,42,+17,31,NaN,1946/47,NaN,NaN
4,5,Sabadell,26,11,8,7,42,36,+6,30,NaN,1946/47,NaN,NaN


In [161]:
# # Check what's at the 1996 URL
# url = build_url_spain(1996)
# print(url)
# response = requests.get(url, headers=HEADERS, timeout=15)
# print(f"Status: {response.status_code}")
# if response.status_code == 200:
#     tables = pd.read_html(io.StringIO(response.text))
#     print(f"{len(tables)} tables found")
#     for i, t in enumerate(tables[:6]):
#         print(f"Table {i}: shape={t.shape}, columns={list(t.columns[:6])}...")


In [162]:
# url = build_url_spain(1996)
# response = requests.get(url, headers=HEADERS, timeout=15)

# try:
#     tables = pd.read_html(io.StringIO(response.text), flavor="lxml")
#     print(f"{len(tables)} tables found")
#     for i, t in enumerate(tables[:6]):
#         print(f"Table {i}: shape={t.shape}, columns={list(t.columns[:6])}...")
# except Exception as e:
#     print(f"ERROR: {e}")



In [163]:
season_1996 = pd.DataFrame({
    "Pos": range(1, 23),
    "Team": [
        "Real Madrid", "Barcelona", "Deportivo La Coruña", "Real Betis",
        "Atlético Madrid", "Athletic Bilbao", "Valladolid", "Real Sociedad",
        "Tenerife", "Valencia", "Compostela", "Espanyol",
        "Racing Santander", "Zaragoza", "Sporting Gijón", "Celta Vigo",
        "Oviedo", "Rayo Vallecano", "Extremadura", "Sevilla",
        "Hércules", "Logroñés"
    ],
    "Pld": [42] * 22,
    "W":  [27,28,21,21,20,16,18,18,15,15,13,14,11,12,13,12,12,13,11,12,12,9],
    "D":  [11, 6,14,14,11,16,10, 9,11,11,14, 9,17,14,11,13,12, 6,11, 7, 5,6],
    "L":  [ 4, 8, 7, 7,11,10,14,15,16,16,15,19,14,16,18,17,18,23,20,23,25,27],
    "GF": [85,102,57,81,76,72,57,50,69,63,52,51,52,58,45,51,49,43,35,50,40,33],
    "GA": [36, 48,30,46,64,57,46,47,57,59,65,57,54,66,63,54,65,62,64,69,77,85],
    "Pts":[92, 90,77,77,71,64,64,63,56,56,53,51,50,50,50,49,48,45,44,43,41,33],
    "Qualification or relegation": [
        "Qualification for the Champions League group stage",
        "Qualification for the Champions League second qualifying round",
        "Qualification for the UEFA Cup first round",
        "Qualification for the Cup Winners' Cup first round",
        "Qualification for the UEFA Cup first round",
        "Qualification for the UEFA Cup first round",
        "Qualification for the UEFA Cup first round",
        None, None, None, None, None, None, None, None, None, None,
        "Qualification for the relegation playoffs",
        None,
        "Relegation to Segunda División",
        "Relegation to Segunda División",
        "Relegation to Segunda División",
    ],
    "season": "1996/97",
})



In [164]:
season_1996

,Pos,Team,Pld,W,D,L,GF,GA,Pts,Qualification or relegation,season
0,1,Real Madrid,42,27,11,4,85,36,92,Qualification for the Champions League group s...,1996/97
1,2,Barcelona,42,28,6,8,102,48,90,Qualification for the Champions League second ...,1996/97
2,3,Deportivo La Coruña,42,21,14,7,57,30,77,Qualification for the UEFA Cup first round,1996/97
3,4,Real Betis,42,21,14,7,81,46,77,Qualification for the Cup Winners' Cup first r...,1996/97
4,5,Atlético Madrid,42,20,11,11,76,64,71,Qualification for the UEFA Cup first round,1996/97
5,6,Athletic Bilbao,42,16,16,10,72,57,64,Qualification for the UEFA Cup first round,1996/97
6,7,Valladolid,42,18,10,14,57,46,64,Qualification for the UEFA Cup first round,1996/97
7,8,Real Sociedad,42,18,9,15,50,47,63,NaN,1996/97
8,9,Tenerife,42,15,11,16,69,57,56,NaN,1996/97
9,10,Valencia,42,15,11,16,63,59,56,NaN,1996/97


In [165]:
df_spain = pd.concat([df_spain, season_1996], ignore_index=True)
df_spain = df_spain.sort_values(["season", "Pos"]).reset_index(drop=True)
print(f"Spain now: {len(df_spain)} rows, {df_spain['season'].nunique()} seasons")


Spain now: 1464 rows, 80 seasons


In [166]:
df_spain['season'].unique()

<StringArray>
[  '1946/47',   '1947/48',   '1948/49',   '1949/50',   '1950/51',   '1951/52',
   '1952/53',   '1953/54',   '1954/55',   '1955/56',   '1956/57',   '1957/58',
   '1958/59',   '1959/60',   '1960/61',   '1961/62',   '1962/63',   '1963/64',
   '1964/65',   '1965/66',   '1966/67',   '1967/68',   '1968/69',   '1969/70',
   '1970/71',   '1971/72',   '1972/73',   '1973/74',   '1974/75',   '1975/76',
   '1976/77',   '1977/78',   '1978/79',   '1979/80',   '1980/81',   '1981/82',
   '1982/83',   '1983/84',   '1984/85',   '1985/86',   '1986/87',   '1987/88',
   '1988/89',   '1989/90',   '1990/91',   '1991/92',   '1992/93',   '1993/94',
   '1994/95',   '1995/96',   '1996/97',   '1997/98',   '1998/99', '1999/2000',
   '2000/01',   '2001/02',   '2002/03',   '2003/04',   '2004/05',   '2005/06',
   '2006/07',   '2007/08',   '2008/09',   '2009/10',   '2010/11',   '2011/12',
   '2012/13',   '2013/14',   '2014/15',   '2015/16',   '2016/17',   '2017/18',
   '2018/19',   '2019/20',   '2020/21'

In [168]:
df_spain['Team'].nunique()

166

In [169]:
np.sort(df_spain['Team'].unique())

array(['Alavés', 'Alavés (R)', 'Alavés[b]', 'Albacete', 'Albacete (O)',
       'Albacete (R)', 'Albacete[c]', 'Alcoyano', 'Alcoyano (R)',
       'Almería', 'Almería (R)', 'Almería[g] (R)', 'Athletic Bilbao',
       'Athletic Bilbao (C)', 'Athletic Bilbao[a]', 'Atlético Bilbao',
       'Atlético Bilbao (C)', 'Atlético Madrid', 'Atlético Madrid (C)',
       'Atlético Madrid (R)', 'Atlético Madrid[a]', 'Atlético Tetuán (R)',
       'Barcelona', 'Barcelona (C)', 'Burgos', 'Burgos (R)', 'Castellón',
       'Castellón (R)', 'Celta', 'Celta (R)', 'Celta Vigo',
       'Celta Vigo (O)', 'Celta Vigo (R)', 'Celta Vigo[c]', 'Compostela',
       'Compostela (R)', 'Condal (R)', 'Cultural Leonesa (R)', 'Cádiz',
       'Cádiz (O)', 'Cádiz (R)', 'Córdoba', 'Córdoba (O)', 'Córdoba (R)',
       'Deportivo La Coruña', 'Deportivo La Coruña (C)',
       'Deportivo La Coruña (O)', 'Deportivo La Coruña (R)', 'Eibar',
       'Eibar (R)', 'Elche', 'Elche (O)', 'Elche (R)', 'Elche[d] (R)',
       'Espanyol', 'Es

1. Burgos CF was founded in 1936, also known as Gimnástica Burgalesa Club de Fútbol. In 1983, the club disappeared due to serious economic problems and the reserve team, Burgos Promesas, was renamed Real Burgos Club de Fútbol.

2. Club Deportivo Condal was a Spanish football club based in Barcelona, in the autonomous community of Catalonia. Founded in 1934 and dissolved in 1970, it held home games at Camp de Les Corts, with a capacity of 25,000 spectators. They played one season in 1956-57


In [173]:
df_spain[df_spain['Team'].str.contains('Condal')]

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts,Qualification or relegation,season,Relegation,Qualification
167,16,Condal (R),30,7,8,15,37,57,−20,22,Relegation to the Segunda División,1956/57,NaN,NaN


In [174]:
SPAIN_NAME_MAP = {
    # Old names → modern name
    "Atlético Bilbao": "Athletic Bilbao",
    "Español": "Espanyol",
    "Celta": "Celta Vigo",
    "RC Deportivo": "Deportivo La Coruña",
    "Recreativo Huelva": "Recreativo",
    "Real Gijón": "Sporting Gijón",
    "Racing": "Racing Santander",
    "Real Santander": "Racing Santander",
    "Gimnástico": "Gimnàstic",
    "Lérida": "Lleida",
    "Real Oviedo": "Oviedo",
    "Real Burgos": "Burgos",
}

SPAIN_CITY_MAP = {
    "Alavés": "Vitoria-Gasteiz",
    "Albacete": "Albacete",
    "Alcoyano": "Alcoy",
    "Almería": "Almería",
    "Athletic Bilbao": "Bilbao",
    "Atlético Madrid": "Madrid",
    "Atlético Tetuán": "Tetuán",
    "Barcelona": "Barcelona",
    "Burgos": "Burgos",
    "Castellón": "Castellón de la Plana",
    "Celta Vigo": "Vigo",
    "Compostela": "Santiago de Compostela",
    "Condal": "Barcelona",
    "Cultural Leonesa": "León",
    "Cádiz": "Cádiz",
    "Córdoba": "Córdoba",
    "Deportivo La Coruña": "A Coruña",
    "Eibar": "Eibar",
    "Elche": "Elche",
    "Espanyol": "Barcelona",
    "Extremadura": "Almendralejo",
    "Getafe": "Getafe",
    "Gimnàstic": "Tarragona",
    "Girona": "Girona",
    "Granada": "Granada",
    "Huesca": "Huesca",
    "Hércules": "Alicante",
    "Jaén": "Jaén",
    "Las Palmas": "Las Palmas de Gran Canaria",
    "Leganés": "Leganés",
    "Levante": "Valencia",
    "Lleida": "Lleida",
    "Logroñés": "Logroño",
    "Mallorca": "Palma",
    "Murcia": "Murcia",
    "Málaga": "Málaga",
    "Mérida": "Mérida",
    "Numancia": "Soria",
    "Osasuna": "Pamplona",
    "Oviedo": "Oviedo",
    "Pontevedra": "Pontevedra",
    "Racing Santander": "Santander",
    "Rayo Vallecano": "Madrid",
    "Real Betis": "Sevilla",
    "Real Burgos": "Burgos",
    "Real Madrid": "Madrid",
    "Real Sociedad": "San Sebastián",
    "Recreativo": "Huelva",
    "Sabadell": "Sabadell",
    "Salamanca": "Salamanca",
    "Sevilla": "Sevilla",
    "Sporting Gijón": "Gijón",
    "Tenerife": "Santa Cruz de Tenerife",
    "Valencia": "Valencia",
    "Valladolid": "Valladolid",
    "Villarreal": "Villarreal",
    "Xerez": "Jerez de la Frontera",
    "Zaragoza": "Zaragoza",
}

def clean_spain_team(name: str) -> str:
    if pd.isna(name):
        return name
    name = SPAIN_NAME_MAP.get(name, name)
    name = re.sub(r'\[.*?\]', '', name)
    name = re.sub(r'\(.*?\)', '', name)
    name = name.strip()
    return SPAIN_NAME_MAP.get(name, name)

df_spain["Team"] = df_spain["Team"].apply(clean_spain_team)
df_spain["GD"] = df_spain["GF"] - df_spain["GA"]
df_spain["GAv"] = (df_spain["GF"] / df_spain["GA"]).round(3)
df_spain["City"] = df_spain["Team"].map(SPAIN_CITY_MAP)

unmapped = df_spain[df_spain["City"].isna()]["Team"].unique()
if len(unmapped):
    print(f"Unmapped teams: {unmapped}")

In [175]:
df_spain.to_csv(Path("data/processed") / "spain_la_liga_standings.csv", index=False)
print(f"Spain saved: {len(df_spain)} rows, {df_spain['season'].nunique()} seasons")


Spain saved: 1464 rows, 80 seasons


In [4]:
df_spain = pd.read_csv(Path("data/processed") / "spain_la_liga_standings.csv")
df_spain.head()

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts,Qualification or relegation,season,Relegation,Qualification,GAv,City
0,1,Valencia,26,16,2,8,54,34,20,34[a],NaN,1946/47,NaN,NaN,1.588,Valencia
1,2,Athletic Bilbao,26,15,4,7,72,38,34,34[a],NaN,1946/47,NaN,NaN,1.895,Bilbao
2,3,Atlético Madrid,26,13,6,7,58,44,14,32,NaN,1946/47,NaN,NaN,1.318,Madrid
3,4,Barcelona,26,14,3,9,59,42,17,31,NaN,1946/47,NaN,NaN,1.405,Barcelona
4,5,Sabadell,26,11,8,7,42,36,6,30,NaN,1946/47,NaN,NaN,1.167,Sabadell


In [8]:
compute_league_winners(df_spain)['City'].value_counts()

City
Madrid           43
Barcelona        27
Valencia          4
Bilbao            3
San Sebastián     2
A Coruña          1
Name: count, dtype: int64

In [9]:
70/80

0.875

In [12]:
def build_url_france(season_start: int) -> str:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    if season_start < 2002:
        page = f"{season_start}%E2%80%93{suffix}_French_Division_1"
    else:
        page = f"{season_start}%E2%80%93{suffix}_Ligue_1"
    return f"https://en.wikipedia.org/wiki/{page}"

test_seasons_france = {
    "Early": 1965,
    "Mid": 1980,
    "Around 2000": 1999,
    "Pre-cutoff": 2001,
    "Post-cutoff": 2003,
    "Recent": 2023,
}

for label, year in test_seasons_france.items():
    url = build_url_france(year)
    print(f"\n{'='*60}")
    print(f"{label} — {year}: {url}")
    response = requests.get(url, headers=HEADERS, timeout=15)
    print(f"  Status: {response.status_code}")
    if response.status_code == 200:
        tables = pd.read_html(io.StringIO(response.text))
        print(f"  {len(tables)} tables found")
        for i, t in enumerate(tables[:6]):
            print(f"  Table {i}: shape={t.shape}, columns={list(t.columns[:6])}...")
    time.sleep(1.5)



Early — 1965: https://en.wikipedia.org/wiki/1965%E2%80%9366_French_Division_1
  Status: 200
  12 tables found
  Table 0: shape=(11, 2), columns=[0, 1]...
  Table 1: shape=(20, 11), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 2: shape=(20, 21), columns=['Home \\ Away', 'ANG', 'BOR', 'CAN', 'RCL', 'LIL']...
  Table 3: shape=(4, 16), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 4: shape=(10, 4), columns=['Rank', 'Player', 'Club', 'Goals']...
  Table 5: shape=(20, 3), columns=['#', 'Club', 'Average attendance']...

Mid — 1980: https://en.wikipedia.org/wiki/1980%E2%80%9381_French_Division_1
  Status: 200
  11 tables found
  Table 0: shape=(11, 2), columns=[0, 1]...
  Table 1: shape=(20, 11), columns=['Pos', 'Team', 'Pld', 'W', 'D', 'L']...
  Table 2: shape=(20, 21), columns=['Home \\ Away', 'ANG', 'AUX', 'BAS', 'BOR', 'LVL']...
  Table 3: shape=(1, 5), columns=['Team 1', 'Agg.Tooltip Aggregate score', 'Team 2', '1st leg', '2nd leg']...
  Table 4: shape=(13, 4), col

In [15]:
for label, year in test_seasons_france.items():
    print(f"\n{label} — {year}")
    try:
        url = build_url_france(year)
        response = requests.get(url, headers=HEADERS, timeout=15)
        tables = pd.read_html(io.StringIO(response.text))
        standings = find_standings_table(tables, str(year))
        if standings is not None:
            print(standings[["Pos", "Team", "Pld", "Pts"]].head(3).to_string(index=False))
            print(f"  {len(standings)} teams")
        else:
            print("  no standings table found")
    except Exception as e:
        print(f"  ERROR: {e}")
    time.sleep(1.5)



Early — 1965
 Pos         Team  Pld  Pts
   1   Nantes (C)   38   60
   2     Bordeaux   38   53
   3 Valenciennes   38   52
  20 teams

Mid — 1980
 Pos              Team  Pld  Pts
   1 Saint-Étienne (C)   38   57
   2            Nantes   38   55
   3          Bordeaux   38   49
  20 teams

Around 2000 — 1999
 Pos                Team  Pld  Pts
   1          Monaco (C)   34   65
   2 Paris Saint-Germain   34   58
   3                Lyon   34   56
  18 teams

Pre-cutoff — 2001
 Pos     Team  Pld  Pts
   1 Lyon (C)   34   66
   2     Lens   34   64
   3  Auxerre   34   59
  18 teams

Post-cutoff — 2003
 Pos                Team  Pld  Pts
   1            Lyon (C)   38   79
   2 Paris Saint-Germain   38   76
   3              Monaco   38   75
  20 teams

Recent — 2023
 Pos                    Team  Pld Pts
   1 Paris Saint-Germain (C)   34  76
   2                  Monaco   34  67
   3                   Brest   34  61
  18 teams


In [17]:
for year in [1945, 1946, 1947, 1948, 1950, 1955, 1960]:
    url = build_url_france(year)
    response = requests.get(url, headers=HEADERS, timeout=15)
    print(f"{year}: {url} → {response.status_code}")
    time.sleep(1.5)


1945: https://en.wikipedia.org/wiki/1945%E2%80%9346_French_Division_1 → 200
1946: https://en.wikipedia.org/wiki/1946%E2%80%9347_French_Division_1 → 200
1947: https://en.wikipedia.org/wiki/1947%E2%80%9348_French_Division_1 → 200
1948: https://en.wikipedia.org/wiki/1948%E2%80%9349_French_Division_1 → 200
1950: https://en.wikipedia.org/wiki/1950%E2%80%9351_French_Division_1 → 200
1955: https://en.wikipedia.org/wiki/1955%E2%80%9356_French_Division_1 → 200
1960: https://en.wikipedia.org/wiki/1960%E2%80%9361_French_Division_1 → 200


In [21]:
def get_france_table(season_start: int) -> pd.DataFrame | None:
    season_end = season_start + 1
    suffix = "2000" if season_end == 2000 else str(season_end)[-2:]
    season_label = f"{season_start}/{suffix}"
    url = build_url_france(season_start)

    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    try:
        tables = pd.read_html(io.StringIO(response.text))
    except Exception as e:
        raise RuntimeError(f"Failed to parse tables from {url}: {e}") from e

    return find_standings_table(tables, season_label)


def scrape_france_seasons(
    start: int = 1963,
    end: int = 2024,
    output_dir: str = "data/processed",
    sleep_seconds: float = 1.5,
) -> pd.DataFrame:
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    all_tables = []
    failed = []

    for year in range(start, end + 1):
        try:
            print(f"{year}: scraping...", end=" ", flush=True)
            t = get_france_table(year)
            if t is not None:
                all_tables.append(t)
                print(f"OK — {len(t)} rows, columns: {list(t.columns)}")
            else:
                print("no standings table found")
                failed.append(year)
        except Exception as e:
            print(f"ERROR — {e}")
            failed.append(year)

        time.sleep(sleep_seconds)

    if not all_tables:
        print("No data collected.")
        return pd.DataFrame()

    df = pd.concat(all_tables, ignore_index=True)

    csv_path = out_path / "france_ligue1_standings.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved {len(df)} rows to {csv_path}")

    if failed:
        print(f"Failed seasons: {failed}")

    return df


# Run
df_france = scrape_france_seasons(start=1946, end=2025)
df_france.head()


1946: scraping... OK — 20 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Qualification or relegation', 'season']
1947: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Qualification or relegation', 'season']
1948: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Qualification or relegation', 'season']
1949: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Qualification or relegation', 'season']
1950: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Qualification or relegation', 'season']
1951: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Qualification or relegation', 'season']
1952: scraping... OK — 18 rows, columns: ['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GAv', 'Pts', 'Qualification or r

,Pos,Team,Pld,W,D,L,GF,GA,GAv,Pts,Qualification or relegation,season,GD,BP,PPG
0,1,Roubaix-Tourcoing (C),38,24,5,9,71,47,1.511,53,NaN,1946/47,NaN,NaN,NaN
1,2,Reims,38,22,5,11,72,40,1.800,49,NaN,1946/47,NaN,NaN,NaN
2,3,Strasbourg,38,21,7,10,79,50,1.580,49,NaN,1946/47,NaN,NaN,NaN
3,4,Lille,38,20,7,11,89,52,1.712,47,NaN,1946/47,NaN,NaN,NaN
4,5,Stade Français,38,19,8,11,72,58,1.241,46,NaN,1946/47,NaN,NaN,NaN


In [22]:
season_2021 = pd.DataFrame({
    "Pos": range(1, 21),
    "Team": [
        "Paris Saint-Germain", "Marseille", "Monaco", "Rennes",
        "Nice", "Strasbourg", "Lens", "Lyon",
        "Nantes", "Lille", "Brest", "Reims",
        "Montpellier", "Angers", "Troyes", "Lorient",
        "Clermont", "Saint-Étienne", "Metz", "Bordeaux"
    ],
    "Pld": [38] * 20,
    "W":  [26, 21, 20, 20, 20, 17, 17, 17, 15, 14, 13, 11, 12, 10,  9,  8,  9,  7,  6,  6],
    "D":  [ 8,  8,  9,  6,  7, 12, 11, 11, 10, 13,  9, 13,  7, 11, 11, 12,  9, 11, 13, 13],
    "L":  [ 4,  9,  9, 12, 11,  9, 10, 10, 13, 11, 16, 14, 19, 17, 18, 18, 20, 20, 19, 19],
    "GF": [90, 63, 65, 82, 52, 60, 62, 66, 55, 48, 49, 43, 49, 44, 37, 35, 38, 42, 35, 52],
    "GA": [36, 38, 40, 40, 36, 43, 48, 51, 48, 48, 57, 44, 61, 55, 53, 63, 69, 77, 69, 91],
    "Pts":[86, 71, 69, 66, 66, 63, 62, 61, 55, 55, 48, 46, 43, 41, 38, 36, 36, 32, 31, 31],
    "Qualification or relegation": [
        "Qualification for the Champions League group stage",
        "Qualification for the Champions League group stage",
        "Qualification for the Champions League third qualifying round",
        "Qualification for the Europa League group stage",
        "Qualification for the Europa Conference League play-off round",
        None, None, None,
        "Qualification for the Europa League group stage",
        None, None, None, None, None, None, None, None,
        "Qualification for the relegation play-offs",
        "Relegation to Ligue 2",
        "Relegation to Ligue 2",
    ],
    "season": "2021/22",
})

df_france = pd.concat([df_france, season_2021], ignore_index=True)
df_france = df_france.sort_values(["season", "Pos"]).reset_index(drop=True)
print(f"France now: {len(df_france)} rows, {df_france['season'].nunique()} seasons")


France now: 1554 rows, 80 seasons


In [23]:
df_france.isna().sum()

Pos                               0
Team                              0
Pld                               0
W                                 0
D                                 0
L                                 0
GF                                0
GA                                0
GAv                            1256
Pts                               0
Qualification or relegation     915
season                            0
GD                              318
BP                             1494
PPG                            1534
dtype: int64

In [24]:
df_france['Team'].nunique()

164

In [25]:
np.sort(df_france['Team'].unique())

array(['Aix-en-Provence (R)', 'Ajaccio', 'Ajaccio (R)', 'Alès',
       'Alès (R)', 'Amiens', 'Amiens (R)', 'Angers', 'Angers (R)',
       'Angoulême', 'Angoulême (R)', 'Arles-Avignon (R)', 'Auxerre',
       'Auxerre (C)', 'Auxerre (R)', 'Auxerre[b]', 'Avignon (R)',
       'Bastia', 'Bastia (D, R)', 'Bastia (O)', 'Bastia (R)', 'Bordeaux',
       'Bordeaux (C)', 'Bordeaux (R)', 'Boulogne (R)', 'Brest',
       'Brest (R)', 'Béziers Hérault (R)', 'Caen', 'Caen (R)', 'Cannes',
       'Cannes (R)', 'Châteauroux (R)', 'Clermont', 'Clermont (R)',
       'Colmar (R)', 'Dijon', 'Dijon (O)', 'Dijon (R)', 'Evian',
       'Evian (R)', 'Gazélec Ajaccio (R)', 'Grenoble', 'Grenoble (R)',
       'Gueugnon (R)', 'Guingamp', 'Guingamp (R)', 'Istres (R)', 'Laval',
       'Laval (R)', 'Le Havre', 'Le Havre (R)', 'Le Mans', 'Le Mans (R)',
       'Lens', 'Lens (C)', 'Lens (O)', 'Lens (R)', 'Lens[b] (D, R)',
       'Lille', 'Lille (C)', 'Lille (O)', 'Lille (R)', 'Limoges',
       'Limoges (R)', 'Lorient', 'Lo

In [ ]:
FRANCE_NAME_MAP = {
    "Paris\xa0Saint-Germain": "Paris Saint-Germain",
    "Matra Racing": "Racing Paris",
    "Troyes-Savinienne": "Troyes",
}

FRANCE_CITY_MAP = {
    "Aix-en-Provence": "Aix-en-Provence",
    "Ajaccio": "Ajaccio",
    "Alès": "Alès",
    "Amiens": "Amiens",
    "Angers": "Angers",
    "Angoulême": "Angoulême",
    "Arles-Avignon": "Arles",
    "Auxerre": "Auxerre",
    "Avignon": "Avignon",
    "Bastia": "Bastia",
    "Bordeaux": "Bordeaux",
    "Boulogne": "Boulogne-sur-Mer",
    "Brest": "Brest",
    "Béziers Hérault": "Béziers",
    "Caen": "Caen",
    "Cannes": "Cannes",
    "Châteauroux": "Châteauroux",
    "Clermont": "Clermont-Ferrand",
    "Colmar": "Colmar",
    "Dijon": "Dijon",
    "Evian": "Évian-les-Bains",
    "Gazélec Ajaccio": "Ajaccio",
    "Grenoble": "Grenoble",
    "Gueugnon": "Gueugnon",
    "Guingamp": "Guingamp",
    "Istres": "Istres",
    "Laval": "Laval",
    "Le Havre": "Le Havre",
    "Le Mans": "Le Mans",
    "Lens": "Lens",
    "Lille": "Lille",
    "Limoges": "Limoges",
    "Lorient": "Lorient",
    "Lyon": "Lyon",
    "Marseille": "Marseille",
    "Martigues": "Martigues",
    "Metz": "Metz",
    "Monaco": "Monaco",
    "Montpellier": "Montpellier",
    "Mulhouse": "Mulhouse",
    "Nancy": "Nancy",
    "Nantes": "Nantes",
    "Nice": "Nice",
    "Niort": "Niort",
    "Nîmes": "Nîmes",
    "Paris FC": "Paris",
    "Paris Saint-Germain": "Paris",
    "RC Paris-Sedan": "Paris",
    "Racing Paris": "Paris",
    "Red Star": "Paris",
    "Reims": "Reims",
    "Rennes": "Rennes",
    "Roubaix-Tourcoing": "Roubaix",
    "Rouen": "Rouen",
    "Saint-Étienne": "Saint-Étienne",
    "Sedan": "Sedan",
    "Sochaux": "Montbéliard",
    "Stade Français": "Paris",
    "Stade Français-Red Star": "Paris",
    "Strasbourg": "Strasbourg",
    "Sète": "Sète",
    "Toulon": "Toulon",
    "Toulouse": "Toulouse",
    "Tours": "Tours",
    "Troyes": "Troyes",
    "Valenciennes": "Valenciennes",
}

def clean_france_team(name: str) -> str:
    if pd.isna(name):
        return name
    name = FRANCE_NAME_MAP.get(name, name)
    name = re.sub(r'\[.*?\]', '', name)
    name = re.sub(r'\(.*?\)', '', name)
    name = name.strip()
    return FRANCE_NAME_MAP.get(name, name)

df_france["Team"] = df_france["Team"].apply(clean_france_team)
df_france["GD"] = df_france["GF"] - df_france["GA"]
df_france["GAv"] = (df_france["GF"] / df_france["GA"]).round(3)
df_france["City"] = df_france["Team"].map(FRANCE_CITY_MAP)

unmapped = df_france[df_france["City"].isna()]["Team"].unique()
if len(unmapped):
    print(f"Unmapped teams: {unmapped}")

df_france.to_csv(Path("data/processed") / "france_ligue1_standings.csv", index=False)
print(f"France saved: {len(df_france)} rows, {df_france['season'].nunique()} seasons")


France saved: 1554 rows, 80 seasons


: 

In [3]:
df_france = pd.read_csv(Path("data/processed") / "france_ligue1_standings.csv")
compute_league_winners(df_france)['Team'].value_counts()

Team
Paris Saint-Germain    14
Saint-Étienne          10
Marseille               9
Monaco                  8
Nantes                  8
Lyon                    7
Reims                   6
Bordeaux                6
Nice                    4
Lille                   3
Roubaix-Tourcoing       1
Strasbourg              1
Auxerre                 1
Lens                    1
Montpellier             1
Name: count, dtype: int64

In [4]:
compute_league_winners(df_france)['Team'].value_counts().sum()

np.int64(80)

In [5]:
compute_league_winners(df_france)['City'].value_counts()

City
Paris            14
Saint-Étienne    10
Marseille         9
Monaco            8
Nantes            8
Lyon              7
Reims             6
Bordeaux          6
Nice              4
Lille             3
Roubaix           1
Strasbourg        1
Auxerre           1
Lens              1
Montpellier       1
Name: count, dtype: int64